<a href="https://colab.research.google.com/github/b2220356179/finetune/blob/main/last_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
from huggingface_hub import notebook_login
notebook_login()


In [15]:
huggingface_user = "zeynepcetin"

In [16]:
# Fine-tuned model name
new_model1 = "distilbert-base-uncased-zeynepc-5dim"

In [8]:
pip install transformers

In [9]:
# Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification
import tensorflow as tf
import numpy as np
from tensorflow import keras
from keras.optimizers import Adam

# Load Your Dataset
df = pd.read_csv("/content/data_edited.csv")  # Load your CSV file

In [17]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Preprocessing: Ensure Question and Answer are strings and handle missing values
df['Question'] = df['Question'].fillna('').astype(str)
df['Answer;'] = df['Answer;'].fillna('').astype(str)

# Combine "Question" and "Answer" fields for the input text
df['text'] = df['Question'] + ' ' + df['Answer;']

# Check if there are any empty rows in the "text" column
df = df[df['text'].str.strip() != '']

# Encode the target into multiple binary labels
df['extraversion'] = df['Target Personality'].apply(lambda x: 1 if x == 'extraversion' else 0)
df['agreeableness'] = df['Target Personality'].apply(lambda x: 1 if x == 'agreeableness' else 0)
df['neuroticism'] = df['Target Personality'].apply(lambda x: 1 if x == 'neuroticism' else 0)
df['openness'] = df['Target Personality'].apply(lambda x: 1 if x == 'openness' else 0)
df['conscientiousness'] = df['Target Personality'].apply(lambda x: 1 if x == 'conscientiousness' else 0)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df[['extraversion', 'agreeableness', 'neuroticism', 'openness', 'conscientiousness']],
    test_size=0.2,
    random_state=42
)

# Verify that X_train and X_test contain only strings
X_train = X_train.astype(str)
X_test = X_test.astype(str)

# Tokenize Input Data
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=512, return_tensors="tf")
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=512, return_tensors="tf")

# Convert into TensorFlow Datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), y_train.values))
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), y_test.values))

# Batch and Shuffle Datasets
train_dataset = train_dataset.batch(16).shuffle(len(X_train))
test_dataset = test_dataset.batch(16)

model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)

# Compile the Model for Multi-label Classification
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train the Model
model.fit(train_dataset, epochs=5, validation_data=test_dataset)

# Evaluate the Model
model.evaluate(test_dataset)

# Make Predictions for each personality dimension
predictions = model.predict(test_dataset)
y_pred = tf.nn.sigmoid(predictions.logits)  # Sigmoid activation to get probabilities
y_pred_binary = np.where(y_pred > 0.5, 1, 0)  # Convert probabilities to binary predictions

# Calculate ROC-AUC score for each personality dimension (optional)
roc_auc = roc_auc_score(y_test, y_pred_binary, average=None)
print(f"ROC-AUC Scores: {roc_auc}")

# Confusion Matrix for each personality trait
for i, trait in enumerate(['extraversion', 'agreeableness', 'neuroticism', 'openness', 'conscientiousness']):
    cm = confusion_matrix(y_test.iloc[:, i], y_pred_binary[:, i])
    print(f"Confusion Matrix for {trait}:")
    print(cm)

# Save trained model
model.save_pretrained("new_model1")
tokenizer.save_pretrained("new_model1")



Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/5
6/6 [==============================] - 47s 5s/step - loss: 0.6492 - accuracy: 0.2500 - val_loss: 0.5614 - val_accuracy: 0.2917
Epoch 2/5
6/6 [==============================] - 32s 6s/step - loss: 0.5279 - accuracy: 0.3958 - val_loss: 0.5031 - val_accuracy: 0.3750
Epoch 3/5
6/6 [==============================] - 30s 5s/step - loss: 0.4732 - accuracy: 0.5729 - val_loss: 0.4586 - val_accuracy: 0.5417
Epoch 4/5
6/6 [==============================] - 30s 5s/step - loss: 0.4189 - accuracy: 0.7500 - val_loss: 0.3998 - val_accuracy: 0.6667
Epoch 5/5
2/2 [==============================] - 3s 566ms/step
ROC-AUC Scores: [0.875 0.8   0.5   0.5   1.   ]
Confusion Matrix for extraversion:
[[16  0]
 [ 2  6]]
Confusion Matrix for agreeableness:
[[19  0]
 [ 2  3]]
Confusion Matrix for neuroticism:
[[23  0]
 [ 1  0]]
Confusion Matrix for openness:
[[16  0]
 [ 8  0]]
Confusion Matrix for conscientiousness:
[[22  0]
 [ 0  2]]


FileNotFoundError: [Errno 2] No such file or directory: 'distilbert-base-uncased-zeynepc-5dim'

In [18]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face Hub
login()

# Verify local directory for saved model
local_dir = "new_model1"
if not os.path.exists(local_dir):
    print(f"Saving model and tokenizer to '{local_dir}'...")
    model.save_pretrained(local_dir)
    tokenizer.save_pretrained(local_dir)
else:
    print(f"Directory '{local_dir}' already exists. Contents:", os.listdir(local_dir))

# Push to the Hugging Face Hub
model.push_to_hub("distilbert-base-uncased-zeynepc-5dim")
tokenizer.push_to_hub("distilbert-base-uncased-zeynepc-5dim")

Directory 'new_model1' already exists. Contents: ['tf_model.h5', 'vocab.txt', 'tokenizer.json', 'special_tokens_map.json', 'config.json', 'tokenizer_config.json']


tf_model.h5:   0%|          | 0.00/268M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/zeynepcetin/distilbert-base-uncased-zeynepc-5dim/commit/366ae472b7e2501a4571846ea07b7dd9c33ba4c4', commit_message='Upload tokenizer', commit_description='', oid='366ae472b7e2501a4571846ea07b7dd9c33ba4c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/zeynepcetin/distilbert-base-uncased-zeynepc-5dim', endpoint='https://huggingface.co', repo_type='model', repo_id='zeynepcetin/distilbert-base-uncased-zeynepc-5dim'), pr_revision=None, pr_num=None)